In [ ]:
%load_ext dotenv
%dotenv

In [1]:
from graphrag.graph_utils import get_neo4j_driver, init_neo4j_constraints, save_chapter, insert_chunks 
from graphrag.war_and_peace_schemas import get_schema_entities, entity_types_map, relationship_classes

import json
import requests

from tqdm import tqdm
from typing import List, Dict, Optional, Literal, Annotated, Union
from pydantic import BaseModel, Field, ConfigDict

neo4j_driver = get_neo4j_driver()

In [2]:
records, summary, keys = neo4j_driver.execute_query(
    """
    MATCH (n)
    UNWIND labels(n) AS node_type
    WITH node_type, count(*) AS total
    RETURN node_type AS type, total AS count

    UNION ALL

    MATCH ()-[r]->()
    WITH type(r) AS relationship_type, count(*) AS total
    RETURN relationship_type AS type, total AS count

    ORDER BY type
    """
)

print([record.data() for record in records])

[{'type': 'Location', 'count': 17}, {'type': 'Person', 'count': 52}, {'type': 'Book', 'count': 1}, {'type': 'Chapter', 'count': 5}, {'type': 'Chunk', 'count': 44}, {'type': 'TitleOrRank', 'count': 20}, {'type': 'Concept', 'count': 11}, {'type': 'Family', 'count': 5}, {'type': 'Event', 'count': 3}, {'type': 'MilitaryUnit', 'count': 1}, {'type': 'HAS_CHAPTER', 'count': 5}, {'type': 'HAS_CHUNK', 'count': 44}, {'type': 'HoldsTitleOrRank', 'count': 34}, {'type': 'LocatedIn', 'count': 6}, {'type': 'MENTIONS', 'count': 350}, {'type': 'ParticipatedIn', 'count': 9}, {'type': 'RelatedTo', 'count': 12}]


In [3]:
data, _, _ = neo4j_driver.execute_query(
    """MATCH (n:Person)
WHERE n.name = "Vasíli Kurágin"
RETURN n.evidence_sentences AS sources"""
)
print([el.data()['sources'] for el in data])

[['With these words she greeted Prince Vasíli Kurágin, a man of high rank and importance, who was the first to arrive at her reception.', 'You must excuse me, dear Vicomte,” said Prince Vasíli to the Frenchman, holding him down by the sleeve in a friendly way to prevent his rising.', 'This unfortunate fete at the ambassador’s deprives me of a pleasure, and obliges me to interrupt you. I am very sorry to leave your enchanting party,” said he, turning to Anna Pávlovna.', 'In passing Prince Vasíli seized Pierre’s hand and said to Anna Pávlovna: “Educate this bear for me! He has been staying with me a whole month and this is the first time I have seen him in society.', 'Believe me, Princess, I am ready to do all I can,” answered Prince Vasíli, “but it is difficult for me to ask the Emperor. I should advise you to appeal to Rumyántsev through Prince Golítsyn. That would be the best way.', 'Prince Vasíli', 'Prince Vasíli knew this, and having once realized that if he asked on behalf of all w

In [4]:
data, _, _ = neo4j_driver.execute_query(
    """MATCH (n)-[r]-(m)
       WHERE elementId(n) < elementId(m)
       WITH n, m, collect(r) AS rels, count(r) AS countOfRels
       ORDER BY countOfRels DESC LIMIT 1
       RETURN n.name AS source, m.name AS target, countOfRels, [r IN rels | {type: type(r), properties: properties(r)}] AS relationships
"""
)
print([el.data() for el in data])

[{'source': 'Borís', 'target': 'Anna Mikháylovna Drubetskáya', 'countOfRels': 2, 'relationships': [{'type': 'RelatedTo', 'properties': {'source_quote': 'my son Borís, Prince', 'relation': 'parent of'}}, {'type': 'RelatedTo', 'properties': {'source_quote': 'her son', 'relation': 'son of'}}]}]


### Summarize each Person's profile based on its description, evidence_sentences collected and relationships he/she holds

In [ ]:
from graphrag.llm import get_openai_client, get_summarize_prompt, PERSON_PROFILE_SYSTEM_PROMPT
from community.community_utils import get_people, import_people_summary
import time

client = get_openai_client()
relationship_types = [c.__name__ for c in relationship_classes]
candidates = get_people (neo4j_driver, relationship_types)

summaries = []
for candidate in tqdm(candidates, desc="Summarizing people profile"):
  # print(f"name: {candidate["name"]}\n"
  #       f"description: {candidate["description"]}\n"
  #       f"evidences: {candidate["evidence_list"]}\n"
  #       f"relationships: {candidate["relationships"]}")
  messages = [{"role": "system", "content": PERSON_PROFILE_SYSTEM_PROMPT},
              {"role": "user",
               "content": get_summarize_prompt(candidate["name"], candidate["description"], 
                                               candidate["evidence_list"], candidate["relationships"])}]
  response = client.chat.completions.create(model="gpt-oss:20b",
                                            temperature=0,
                                            reasoning_effort="low",
                                            max_completion_tokens=4096,
                                            messages=messages)
  # print(f"Summary: {response.choices[0].message.content}\n")
  summaries.append({"name": candidate["name"], "summary": response.choices[0].message.content})
  time.sleep(15)
import_people_summary(neo4j_driver, summaries)

In [5]:
summary, _, _ = neo4j_driver.execute_query(
    """MATCH (n:Person)
WHERE n.name = "Andrew Bolkónski"
RETURN n.summary AS summary""")
print(summary[0]["summary"])

Andrew Bolkónski was a young prince of medium height, described as very handsome with firm, clear‑cut features. He held the title of Prince and was married to Lise Meinen, the “little princess.”  Bolkónski had served in the army under the late Emperor and was nicknamed “the King of Prussia.”  Contemporary observers described him as clever but eccentric, sometimes a bore.  In social scenes he was noted for his sarcastic smile, for quoting Napoleon’s words, and for looking Anna straight in the face while engaging in witty banter.


### Identify Communities, groups of nodes more densely connected to each other than to the rest of the graph

In [2]:
from community.community_utils import  calculate_communities

community_distribution = calculate_communities(neo4j_driver)
community_distribution
#print(f"There are {community_distribution['communityCount']} communities with distribution: {community_distribution['communityDistribution']}")

{'communityCount': 4,
 'modularity': 0.6527777777777778,
 'modularities': [0.6527777777777778]}

In [ ]:
from community.community_utils import get_community_info, add_community_info
from community.data import CommunityReport
from community.community_llm import COMMUNITY_REPORT_SYSTEM_PROMPT, COMMUNITY_REPORT_USER_PROMPT

from graphrag.llm import get_openai_client
client = get_openai_client()

community_info = get_community_info(neo4j_driver)
communities = []
for community in tqdm(community_info, desc="Summarizing communities"):
  input_text = json.dumps(community.data(), ensure_ascii=False, indent=2, default=str)
  response = client.chat.completions.create(model="gpt-oss:20b",
                                           temperature=0,
                                           reasoning_effort="low",
                                           max_completion_tokens=4096,
                                           messages=[{"role": "system", "content": COMMUNITY_REPORT_SYSTEM_PROMPT},
                                                     {"role": "user",
                                                      "content": COMMUNITY_REPORT_USER_PROMPT.format(input_text=input_text)}],
                                           response_format=CommunityReport)

  report = response.choices[0].message.parsed
  print(report)
  communities.append({"community": report.model_dump(),
                      "communityId": community["communityId"],
                      "nodes": [el["id"] for el in community["nodes"]]})

result = add_community_info(neo4j_driver, communities)
print(result)

Summarizing communities:  25%|██▌       | 1/4 [01:51<05:34, 111.40s/it]

title='Community\u202f1 – The Kurágin Family at Court' summary='The supplied graph depicts a compact aristocratic family centered on Prince\u202fVasíli\u202fKúrágin and his three children.  The network is dominated by parent‑child links; no other inter‑family or external ties are recorded.  The community’s structure is therefore a simple star with the prince at the hub.  The narrative significance of this cluster is limited to the prince’s social role at a single reception and the financial strain his youngest son imposes.  No political alliances, marriages, or broader social networks are represented.' rating=2.5 rating_explanation='The community contains only four individuals and a handful of explicit relationships.  The only events mentioned are a single reception and a comment on Anatole’s spending.  These facts suggest the cluster is a minor subplot within a larger narrative, hence a limited significance score of 2.5.' findings=[CommunityFinding(summary='Centrality of Prince\u202fV

Summarizing communities:  50%|█████     | 2/4 [02:54<02:45, 82.73s/it] 

title='Community\u202f5 – The Bolkónski–Meinen Marriage' summary='Community\u202f5 consists of a single marital pair: Lise\u202fMeinen and Andrew\u202fBolkónski.  The graph contains only two person nodes and two reciprocal “married to” relationships, both explicitly supported by the supplied text.  No other entities, relationships, or contextual details are present in the data set.' rating=3.5 rating_explanation='The community is small and contains only a single, straightforward relationship.  While the individuals are described in some detail, the graph itself offers no additional connections, events, or narrative functions beyond the marital tie.  Consequently the community’s significance in the supplied material is limited but not negligible, placing it in the 2.1–4.0 range.' findings=[CommunityFinding(summary='The community is defined solely by a marital bond between Lise\u202fMeinen and Andrew\u202fBolkónski.', explanation='Both nodes are linked by two reciprocal “RelatedTo” edges

Summarizing communities:  75%|███████▌  | 3/4 [05:21<01:52, 112.47s/it]

title='Community\u202f7 – Count\u202fBezúkhov and his illegitimate son Pierre' summary='This community consists of two individuals linked by a single parent‑child relationship. The graph contains no additional social, political, or professional ties, so the community’s structure is essentially a dyad. The narrative significance of this dyad is limited to the brief mention of a familial connection that situates Pierre within the aristocratic milieu of Catherine\u202fthe\u202fGreat’s Russia.' rating=4.0 rating_explanation='The community is small and contains only a single explicit relationship. The information supplied is minimal and does not reveal any broader plot‑moving events or interactions beyond the parentage. Consequently, the narrative significance is limited but not negligible, as the relationship establishes Pierre’s social standing and potential motivations in the broader story. The rating of 4.0 reflects this moderate but constrained importance.' findings=[CommunityFinding(s

Summarizing communities: 100%|██████████| 4/4 [06:49<00:00, 102.50s/it]

title='Community\u202f9 – A Small Royal Lineage' summary='Community\u202f9 consists of a three‑node family unit: Prince\u202fBorís, his mother Anna\u202fMikháylovna\u202fDrubetskáya, and Anna’s unnamed father. The only relationships recorded are the parent‑child links between mother and son and between father and mother. No other social, political, or professional ties appear in the supplied data.' rating=2.5 rating_explanation='The community contains only a single familial relationship chain and no evidence of broader influence, political activity, or narrative events beyond the mention of a planned transfer to the Guards. Consequently its narrative significance in the supplied material is limited, though the transfer hint suggests a potential plot point that is not elaborated here.' findings=[CommunityFinding(summary='Centrality of the parent‑child chain', explanation='All three nodes are connected solely through the two parent‑child relationships. This linear structure indicates tha

[{'communitiesProcessed': 4, 'membershipsProcessed': 11}]


In [4]:
data, _, _ = neo4j_driver.execute_query("""MATCH (c:Community)
                                           WITH c, COUNT { (c)<-[:InCommunity]-() } AS size
                                           ORDER BY size DESC
                                           LIMIT 1
                                           RETURN c.communityId AS communityId,
                                                  c.title AS title,
                                                  c.summary AS summary,
                                                  c.rating AS rating,
                                                  c.rating_explanation AS ratingExplanation,
                                                  size AS numberOfMembers""")

if data:
  print(f"Community with more people: {data[0]['title']}")
  print(f"ID: {data[0]['communityId']}")
  print(f"Members: {data[0]['numberOfMembers']}")
  print(f"Rating: {data[0]['rating']}")
  print(data[0]['summary'])
else:
  print("No communities found.")

Community with more people: Community 1 – The Kurágin Family at Court
ID: 1
Members: 4
Rating: 2.5
The supplied graph depicts a compact aristocratic family centered on Prince Vasíli Kúrágin and his three children.  The network is dominated by parent‑child links; no other inter‑family or external ties are recorded.  The community’s structure is therefore a simple star with the prince at the hub.  The narrative significance of this cluster is limited to the prince’s social role at a single reception and the financial strain his youngest son imposes.  No political alliances, marriages, or broader social networks are represented.


### Global search: Use a Map/Reduce strategy to analyse individual knowledge graph community summaries and combine their findings into a comprehensive answer to broader questions.

In [2]:
from community.data import GlobalMapResponse
from community.community_llm import (GLOBAL_MAP_SYSTEM_PROMPT, 
                                     GLOBAL_MAP_USER_PROMPT, 
                                     GLOBAL_REDUCE_SYSTEM_PROMPT, 
                                     GLOBAL_REDUCE_USER_PROMPT)
from graphrag.llm import get_openai_client
client = get_openai_client()
   
def community_search_consolidator (contexts, user_query):
  all_points = []
  for item in tqdm(contexts, desc="Processing communities"):
    # MAP time: process each community independently
    response = client.chat.completions.parse(model="gpt-oss:20b",
                                             temperature=0,
                                             reasoning_effort="low",
                                             max_completion_tokens=4096,
                                             messages=[{"role": "system", "content": GLOBAL_MAP_SYSTEM_PROMPT},
                                                       {"role": "user", 
                                                        "content": GLOBAL_MAP_USER_PROMPT.format(query=user_query,
                                                                                                 community_context=item["summary"])}],
                                             response_format=GlobalMapResponse)

    # Collect relevant points, preserving their provenance
    for point in response.choices[0].message.parsed.points:
        all_points.append({"communityId": item["communityId"],
                           "description": point.description,
                           "score": point.score})

  # REDUCE time: combine all community results
  if not all_points:
    return "No relevant information was found in the available communities."
  report_data = json.dumps(all_points, ensure_ascii=False, indent=2)        
  response = client.chat.completions.create(model="gpt-oss:20b",
                                            temperature=0,
                                            reasoning_effort="low",
                                            max_completion_tokens=4096,
                                            messages=[{"role": "system", "content": GLOBAL_REDUCE_SYSTEM_PROMPT},
                                                      {"role": "user", 
                                                       "content": GLOBAL_REDUCE_USER_PROMPT.format(query=user_query,
                                                                                                   report_data=report_data)}])
  return response.choices[0].message.content

In [3]:
import textwrap
from community.community_utils import global_searcher

query = "Who is Borís?"
print(textwrap.fill(global_searcher(neo4j_driver, community_search_consolidator, query, rating_threshold = 0), width=120))

Processing communities: 100%|██████████| 4/4 [02:02<00:00, 30.64s/it]


**Borís**  - Borís is identified as a **prince**.   - He is the **son of Anna Mikháylovna Drubetskáya**.   - His
grandfather is Anna’s unnamed father, placing him within a three‑node family unit.


In [ ]:
query = "What is this story about?"
print(textwrap.fill(global_searcher(neo4j_driver, community_search_consolidator, query, rating_threshold = 3), width=120))

global_searcher:: Retrieved 2 community reports


Processing communities: 100%|██████████| 2/2 [01:46<00:00, 53.23s/it]


**Overview of the story**

The narrative is built around two distinct interpersonal relationships that anchor the plot:

1. **Count Bezúkhov and his illegitimate son Pierre**  
   - The central focus is the father‑child bond between Count Bezúkhov and Pierre, who is his illegitimate son.  
   - This relationship places Pierre within the aristocratic milieu of Catherine the Great’s Russia, hinting at the social dynamics and possible motivations that drive the broader storyline.  
   - *[Data: Communities (7)]*

2. **Lise Meinen and Andrew Bolkónski**  
   - The story also includes a marital relationship between Lise Meinen and Andrew Bolkónski.  
   - No additional characters, events, or narrative details are provided for this pairing.  
   - *[Data: Communities (5)]*

**What the story is about**

The narrative explores the complexities of family ties and social status in imperial Russia, primarily through the lens of Count Bezúkhov’s relationship with his son Pierre. It also touches on

### Build embedding on Person's entity summary field

In [3]:
from sentence_transformers import SentenceTransformer
from community.community_utils import retrieve_people, store_people_encoding

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)

def embed_text(text: str):
  return model.encode([text], normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)

people = retrieve_people(neo4j_driver)
people = [{"name": person["name"], "embedding": embed_text(person["summary"])[0]} for person in people]
store_people_encoding(neo4j_driver, people, model.get_sentence_embedding_dimension())

Batches: 100%|██████████| 1/1 [00:00<00:00, 19.65it/s]
/tmp/ipykernel_305035/4291702382.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  store_people_encoding(neo4j_driver, people, model.get_sentence_embedding_dimension())


### Local search: Retrieves specific entities, relationships and text passages relevant to a user's question using semantic and graph-based retrieval

In [ ]:
from community.community_utils import local_search
from community.community_llm import LOCAL_SEARCH_SYSTEM_PROMPT, LOCAL_SEARCH_USER_PROMPT

def local_searcher(query: str) -> str:
  k_entities = 5
  topChunks = 3
  topCommunities = 3

  query_embedding = embed_text(query)[0]
  records = local_search(neo4j_driver, query_embedding, k_entities, topChunks, topCommunities)

  if not records:
    print("No relevant information was found.")
  context = records[0]["context"]
  
  if not context or not context["Entities"]:
    print("No relevant characters were found.")

  # Prepare context for the LLM
  context_str = json.dumps(context, ensure_ascii=False, indent=2, default=str)

  # Generate final answer
  messages = [{"role": "system",
                       "content": LOCAL_SEARCH_SYSTEM_PROMPT},
              {"role": "user",
                       "content": LOCAL_SEARCH_USER_PROMPT.format(query=query, context_data=context_str)}]

  response = client.chat.completions.create(model="gpt-oss:20b",
                                            temperature=0,
                                            reasoning_effort="low",
                                            max_completion_tokens=4096,
                                            messages=messages)
  return response.choices[0].message.content


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1903.39it/s]


In [ ]:
import textwrap
#print(local_searcher("Who is Anna Pávlovna Schérer?"))
print(textwrap.fill(local_searcher("Who is Vasili Kurágin"), width=120))

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.35it/s]


{'Relationships': [{'source': 'Vasíli Kurágin', 'relationship': 'RelatedTo', 'properties': {'source_quote': 'I don’t speak of Anatole, your youngest', 'relation': 'parent of'}, 'target': 'Anatole'}, {'source': 'Anatole', 'relationship': 'RelatedTo', 'properties': {'source_quote': 'younger son', 'relation': 'son of'}, 'target': 'Vasíli Kurágin'}], 'Chunks': [{'text': 'When he has been transferred to the Guards...” she faltered. “You are on good terms with Michael Ilariónovich Kutúzov ... recommend Borís to him as adjutant! Then I shall be at rest, and then...” Prince Vasíli smiled. “No, I won’t promise that. You don’t know how Kutúzov is pestered since his appointment as Commander in Chief. He told me himself that all the Moscow ladies have conspired to give him all their sons as adjutants.” “No, but do promise! I won’t let you go! My dear benefactor...” “Papa,” said his beautiful daughter in the same tone as before, “we shall be late.” “Well, au revoir! Good-by! You hear her?” “Then to